# Swin + CLAHE on NIH 14 Chest X-ray

Using CLAHE (Contrast Limited Adaptive Histogram Equalization) preprocessing on the NIH 14 with swin achieved .38% increase in AUC and notably 93.2% accuracy on Emphysema





In [1]:
import sys
print(sys.executable)

c:\Users\nick\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

# using mixed precision training for speed increase
from torch.cuda.amp import GradScaler, autocast


from sklearn.metrics import roc_auc_score


import cv2

class CXR8Dataset(Dataset):
     def __init__(self, df, labels, idx_array, transform, lookup):
         self.df        = df.iloc[idx_array].reset_index(drop=True)
         self.labels    = labels[idx_array]
         self.transform = transform
         self.lookup    = lookup
     def __len__(self):
         return len(self.df)
     def __getitem__(self, i):
         fname = self.df.loc[i, "Image Index"]
         img   = Image.open(self.lookup[fname]).convert("RGB")
         img   = self.transform(img)
         lbl   = torch.tensor(self.labels[i], dtype=torch.float32)
         return img, lbl


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img: Image.Image) -> Image.Image:
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB) # convert to LAB
        lab[:, :, 0] = self.clahe.apply(lab[:, :, 0]) # apply CLAHE
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB) # back to RGB
        return Image.fromarray(enhanced)  


# Train / eval loops
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    with torch.set_grad_enabled(train):
        for imgs, lbls in tqdm(loader, desc="train" if train else "val ", leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)

            if not train:
                all_logits.append(logits.sigmoid().cpu().detach())
                all_labels.append(lbls.cpu().detach())

    avg_loss = total_loss / len(loader.dataset)

    if not train:
        probs  = torch.cat(all_logits).numpy()
        labels = torch.cat(all_labels).numpy()
        aucs = []
        for c in range(labels.shape[1]):
            if labels[:, c].sum() > 0:  # skip classes with no positive examples
                aucs.append(roc_auc_score(labels[:, c], probs[:, c]))
        return avg_loss, np.mean(aucs)

    return avg_loss, None


if __name__ == "__main__":

    # Load labels
    data_frame = pd.read_csv("../chest_xray_dataset/CXR8/Data_Entry_2017_v2020.csv")

    # Keep only the two columns we need
    data_frame = data_frame[["Image Index", "Finding Labels"]].copy()

    # Parse multi-label strings  e.g. "Atelectasis|Cardiomegaly"
    data_frame["labels"] = data_frame["Finding Labels"].str.split("|")

    ALL_CLASSES = [
        "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
        "Effusion", "Emphysema", "Fibrosis", "Hernia",
        "Infiltration", "Mass", "No Finding", "Nodule",
        "Pleural_Thickening", "Pneumonia", "Pneumothorax",
    ]
    NUM_CLASSES = len(ALL_CLASSES)

    mlb = MultiLabelBinarizer(classes=ALL_CLASSES)
    label_matrix = mlb.fit_transform(data_frame["labels"])  # (N, 15)

    # Build image-path index
    IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"

    # Recursively find every PNG once and build a filename -> full-path dict
    all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
    path_lookup = {os.path.basename(p): p for p in all_png}
    print(f"Found {len(path_lookup):,} images on disk.")

    # Filter dataframe to images that actually exist
    mask = data_frame["Image Index"].isin(path_lookup)
    data_frame = data_frame[mask].reset_index(drop=True)
    label_matrix = label_matrix[mask.values]
    print(f"Matched {len(data_frame):,} rows after filtering.")

    # Train / val split
    indices = np.arange(len(data_frame))
    train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

    # Dataset
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    val_tf = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])



    train_ds = CXR8Dataset(data_frame, label_matrix, train_idx, train_tf, path_lookup)
    val_ds   = CXR8Dataset(data_frame, label_matrix, val_idx,   val_tf,   path_lookup)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
    # Replace the classification head for multi-label output
    in_features = model.head.in_features
    model.head  = nn.Linear(in_features, NUM_CLASSES)
    model       = model.to(device)

    # Training setup
    criterion = nn.BCEWithLogitsLoss()          # multi-label
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)


    NUM_EPOCHS = 10
    best_val   = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, _        = run_epoch(train_loader, train=True)
        val_loss, val_auc = run_epoch(val_loader,   train=False)
        scheduler.step()  

        flag = ""
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "swin_cxr8_best.pth")
            flag = "  - saved"
   
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
            f"train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}   "
            f"val_auc={val_auc:.4
                       f}{flag}")

    print("Done. Best val loss:", round(best_val, 4))

Found 112,120 images on disk.
Matched 112,120 rows after filtering.
Train: 95,302  |  Val: 16,818
Device: cuda


Epoch 01/10  train_loss=0.1937  val_loss=0.1815  val_auc=0.8003  - saved


Epoch 02/10  train_loss=0.1817  val_loss=0.1767  val_auc=0.8148  - saved


Epoch 03/10  train_loss=0.1774  val_loss=0.1754  val_auc=0.8206  - saved


Epoch 04/10  train_loss=0.1741  val_loss=0.1752  val_auc=0.8255  - saved


...

Epoch 06/10  train_loss=0.1678  val_loss=0.1721  val_auc=0.8343  - saved


Epoch 07/10  train_loss=0.1646  val_loss=0.1726  val_auc=0.8358


Epoch 08/10  train_loss=0.1615  val_loss=0.1708  val_auc=0.8402  - saved


Epoch 09/10  train_loss=0.1588  val_loss=0.1720  val_auc=0.8387


Epoch 10/10  train_loss=0.1576  val_loss=0.1717  val_auc=0.8393
Done. Best val loss: 0.1708


In [4]:
import pandas as pd
import numpy as np
import os
import glob
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import cv2


class CXR8Dataset(Dataset):
    def __init__(self, df, labels, idx_array, transform, lookup):
        self.df        = df.iloc[idx_array].reset_index(drop=True)
        self.labels    = labels[idx_array]
        self.transform = transform
        self.lookup    = lookup
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        fname = self.df.loc[i, "Image Index"]
        img   = Image.open(self.lookup[fname]).convert("RGB")
        img   = self.transform(img)
        lbl   = torch.tensor(self.labels[i], dtype=torch.float32)
        return img, lbl


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    def __call__(self, img):
        img_np  = np.array(img)
        lab     = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = self.clahe.apply(lab[:, :, 0])
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(enhanced)


ALL_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "No Finding", "Nodule",
    "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]
NUM_CLASSES   = len(ALL_CLASSES)
IMAGE_ROOT    = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"
CHECKPOINT    = "swin_cxr8_best.pth"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Data
data_frame = pd.read_csv("../chest_xray_dataset/CXR8/Data_Entry_2017_v2020.csv")
data_frame = data_frame[["Image Index", "Finding Labels"]].copy()
data_frame["labels"] = data_frame["Finding Labels"].str.split("|")

mlb          = MultiLabelBinarizer(classes=ALL_CLASSES)
label_matrix = mlb.fit_transform(data_frame["labels"])

all_png      = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
path_lookup  = {os.path.basename(p): p for p in all_png}

mask         = data_frame["Image Index"].isin(path_lookup)
data_frame   = data_frame[mask].reset_index(drop=True)
label_matrix = label_matrix[mask.values]

indices      = np.arange(len(data_frame))
_, val_idx   = train_test_split(indices, test_size=0.15, random_state=42)  # same split

val_tf = transforms.Compose([
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_ds     = CXR8Dataset(data_frame, label_matrix, val_idx, val_tf, path_lookup)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# Model
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model      = swin_t(weights=None)   # no pretrained weights, we load our own
in_features = model.head.in_features
model.head  = nn.Linear(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model       = model.to(device)
model.eval()

# Inference
all_logits, all_labels = [], []

with torch.no_grad():
    for imgs, lbls in tqdm(val_loader, desc="eval"):
        imgs = imgs.to(device)
        all_logits.append(model(imgs).sigmoid().cpu())
        all_labels.append(lbls)

probs  = torch.cat(all_logits).numpy()
labels = torch.cat(all_labels).numpy()

# Per-class AUC
print(f"\n{'Class':<22} {'AUC':>6}")
print("-" * 30)

aucs = {}
for c in range(NUM_CLASSES):
    if labels[:, c].sum() > 0:
        aucs[ALL_CLASSES[c]] = roc_auc_score(labels[:, c], probs[:, c])

for cls, auc in sorted(aucs.items(), key=lambda x: x[1]):
    print(f"  {cls:<20} {auc:.3f}")

print("-" * 30)
print(f"  {'Mean AUC':<20} {np.mean(list(aucs.values())):.3f}")

C:\Users\nick\AppData\Local\Temp\ipykernel_19824\3214181035.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(CHECKPOINT, map_location=de


Class                     AUC
------------------------------
  Infiltration         0.723
  Pneumonia            0.765
  Nodule               0.778
  No Finding           0.789
  Pleural_Thickening   0.808
  Consolidation        0.818
  Atelectasis          0.822
  Fibrosis             0.827
  Hernia               0.872
  Mass                 0.874
  Effusion             0.889
  Cardiomegaly         0.898
  Pneumothorax         0.903
  Edema                0.904
  Emphysema            0.932
------------------------------
  Mean AUC             0.840
